In [1]:
import pandas as pd
from ecifp import utils
from collections import defaultdict
from ecifp.fp_sim import get_bs_clusters

In [2]:
## Get the data directory
DATADIR = utils.get_data_dir()

In [3]:
ligands_ignored = []
with open(DATADIR / 'ligands_ignored.txt', 'r') as f:
    for line in f:
        ligands_ignored.append(line.strip())

In [4]:
def get_interacting_residues(uniprot_id, lignads_ignored):
    url = f"https://www.ebi.ac.uk/pdbe/api/v2/uniprot/ligand_sites/{uniprot_id}"
    ligand_sites = utils.fetch_api_data(url)
    intx_residues = defaultdict(list)
    if not ligand_sites:
        return pd.DataFrame([])
    for ligand in ligand_sites[uniprot_id]["data"]:
        ligand_id = ligand["accession"]
        if ligand_id not in ligands_ignored:
            for residue in ligand["residues"]:
                uniprot_residues = list(range(residue['startIndex'], residue['endIndex']+1))
                for intx_entry in residue["interactingPDBEntries"]:
                    intx_residues[f"{intx_entry['pdbId']}_{intx_entry['chainIds']}_{ligand_id}"].extend(uniprot_residues)
    return pd.DataFrame({'bound_molecule': intx_residues.keys(), 'intx_residues':intx_residues.values()})
    

In [5]:
intx_residues = get_interacting_residues("Q47396", ligands_ignored)

In [6]:
data_matrix = (
    pd.get_dummies(
        intx_residues
        .explode('intx_residues')
        ['intx_residues']
    )
    .groupby(level=0)
    .sum()
).values

In [7]:
clusters = get_bs_clusters(data_matrix)

In [8]:
clusters

array([2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2], dtype=int32)